# MOF-74-Mn — Bulk Periodic Structure
## Energy, Forces, Geometry Optimisation and Electronic Structure

---

### What is this notebook about?

This is the **third and most complex level** of the MOF-74-Mn series:

| Notebook | System | Methods |
|---|---|---|
| 1 · `dobdc_ligand_properties.ipynb` | Isolated H₄DOBDC | RDKit, xTB, ORCA (molecule) |
| 2 · `mn_dobdc_complex.ipynb` | Mn-DOBDC cluster | xTB UHF, ORCA mult=6 |
| **3 · This notebook** | **Bulk periodic MOF-74-Mn** | **MACE + xTB with PBC** |

---

### What makes a periodic material different?

In an **isolated molecule** (notebooks 1 & 2), the structure is finite: it has edges, and
properties are described by discrete orbitals (HOMO, LUMO).

In a **crystal** like MOF-74-Mn:
- The structure repeats infinitely in 3-D via **periodic boundary conditions (PBC)**
- The minimum non-repeating unit is the **unit cell**, described by 6 lattice parameters (a, b, c, α, β, γ)
- Molecular orbitals become **bands** — continuous manifolds of states labelled by crystal momentum **k**
- The energy gap between bands is the **band gap** (not HOMO-LUMO gap)
- Properties depend on both the cell geometry and the PBC-wrapped neighbour interactions

---

### MOF-74 (CPO-27) Crystal Structure

```
Space group  :  R-3  (rhombohedral / trigonal, #148)
Topology     :  Hexagonal rod-packing (hxl net)
Metal SBU    :  1-D Mn₂-O₂ chains running along the c-axis
Pore shape   :  1-D hexagonal channels (~11 Å pore diameter)
Pore access  :  One open metal site per Mn (the axial water desorbs)

Approximate cell (hexagonal setting):
  a = b = 26.1 Å   c = 6.8 Å   α = β = 90°  γ = 120°
```

The DOBDC⁴⁻ linker bridges adjacent Mn–O chains, creating a honeycomb of
large pores. Activated MOF-74-Mn is one of the best materials for O₂ and
CO₂ adsorption because of the exposed Mn²⁺ Lewis-acid sites.

---

### Getting the crystal structure (CIF)

A CIF file contains atomic fractional coordinates + lattice parameters.
Set `CIF_PATH` below to your file. Recommended sources:

| Source | URL | Note |
|---|---|---|
| CCDC | https://www.ccdc.cam.ac.uk/ | Search *OWAJIA* (CPO-27-Mn dehydrated) |
| Materials Project | https://materialsproject.org/ | Search *mp-636249* |
| CoRE MOF 2019 | https://doi.org/10.6084/m9.figshare.8 | Full open database |

If no CIF is provided, this notebook builds a **simplified periodic model** (the
Mn-DOBDC cluster from notebook 2 placed in a periodic box) to demonstrate all
workflow steps.  Results from the simplified model are **not physically meaningful**
for MOF-74 science — they only demonstrate the PBC code path.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CIF_PATH        = None   # set to "path/to/mof74_mn.cif" if you have it
                         # e.g. downloaded from CCDC (OWAJIA) or Materials Project

CUTOFF          = 6.0    # neighbour-list cutoff (Å) — larger for periodic MOF
MACE_MODEL      = "small"
ORCA_CMD        = None   # ORCA path (only used for cluster-model section)
WORK_DIR        = "mof74_work"
FMAX            = 0.10   # force convergence for geometry optimisation (eV/Å)
KPOINTS         = None   # for future periodic DFT (CP2K/VASP); not used here


In [ ]:
import os, re, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ase, ase.io
from ase import Atoms
from ase.optimize import BFGS
from ase.neighborlist import NeighborList
from ase.cell import Cell
from goal.ml.data.graph import AtomicGraph

os.makedirs(WORK_DIR, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False

try:
    import py3Dmol; HAS_PY3DMOL = True
except ImportError:
    HAS_PY3DMOL = False

try:
    from xtb.ase.calculator import XTB as XTBCalcASE; HAS_XTB_ASE = True
except ImportError:
    HAS_XTB_ASE = False

try:
    from xtb.interface import Calculator as XTBInterface, Param; HAS_XTB_IFACE = True
except ImportError:
    HAS_XTB_IFACE = False

try:
    from mace.calculators import mace_mp; HAS_MACE = True
except ImportError:
    HAS_MACE = False; print("MACE missing — pip install mace-torch")

try:
    from ase.calculators.orca import ORCA, OrcaProfile, OrcaTemplate; HAS_ORCA = True
except ImportError:
    HAS_ORCA = False

CPK = {
    "H":"#FFFFFF","C":"#808080","N":"#4169E1","O":"#FF4500","S":"#FFD700",
    "Mn":"#9C3E8F","Fe":"#E06000","Co":"#0000C8","Ni":"#28A028",
    "Cu":"#C88033","Mg":"#8AFF00","Ca":"#3DFF00","Zn":"#7D80B0",
}
VDW = {"H":0.31,"C":0.77,"N":0.75,"O":0.73,"Mn":1.61,"Fe":1.56,
       "Co":1.52,"Ni":1.49,"Cu":1.45,"Mg":1.60,"Zn":1.42}
print("Imports done.")


## Section 2 · Crystal Structure — Load or Build

In [ ]:
atoms_mof = None

# ── Option A: Load from CIF ──────────────────────────────────────────────────
if CIF_PATH is not None and os.path.exists(CIF_PATH):
    try:
        atoms_mof = ase.io.read(CIF_PATH)
        print(f"✓ Loaded from CIF: {CIF_PATH}")
        print(f"  Atoms      : {len(atoms_mof)}")
        print(f"  PBC        : {atoms_mof.get_pbc()}")
        lengths = atoms_mof.cell.lengths()
        angles  = atoms_mof.cell.angles()
        print(f"  a,b,c (Å)  : {lengths[0]:.3f}  {lengths[1]:.3f}  {lengths[2]:.3f}")
        print(f"  α,β,γ (°)  : {angles[0]:.2f}  {angles[1]:.2f}  {angles[2]:.2f}")
    except Exception as exc:
        print(f"✗ Could not load CIF: {exc}")
        atoms_mof = None
else:
    if CIF_PATH is None:
        print("CIF_PATH is None — using simplified periodic model")
    else:
        print(f"CIF file not found: {CIF_PATH}")

# ── Option B: Build simplified periodic model ─────────────────────────────────
if atoms_mof is None:
    print()
    print("Building simplified periodic model (Mn-DOBDC in a cubic box).")
    print("NOTE: This is NOT the real MOF-74-Mn crystal structure.")
    print("      It demonstrates the PBC workflow; replace with a real CIF for science.")
    print()

    if HAS_RDKIT:
        SMILES = "OC(=O)c1cc(O)c(C(=O)O)cc1O"
        mol = Chem.AddHs(Chem.MolFromSmiles(SMILES))
        ps = AllChem.ETKDGv3(); ps.randomSeed = 42
        AllChem.EmbedMolecule(mol, ps)
        AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94s")
        conf = mol.GetConformer(0)
        pos_d = np.array(conf.GetPositions())
        sym_d = [a.GetSymbol() for a in mol.GetAtoms()]
    else:
        # Minimal fallback: C₆H₄ benzene ring placeholder
        theta = np.linspace(0, 2*np.pi, 7)[:-1]
        pos_d = np.column_stack([1.4*np.cos(theta), 1.4*np.sin(theta), np.zeros(6)])
        sym_d = ["C"]*6

    dobdc = Atoms(sym_d, positions=pos_d)

    # Identify carboxylate C=O oxygens and phenolate oxygens algorithmically
    carbonyl_o, phenol_o = [], []
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        if atom.GetAtomicNum() == 8:
            nbrs = list(atom.GetNeighbors())
            has_arom_C = any(n.GetAtomicNum()==6 and n.GetIsAromatic() for n in nbrs)
            has_H      = any(n.GetAtomicNum()==1 for n in nbrs)
            has_dbl_C  = any(
                mol.GetBondBetweenAtoms(idx, n.GetIdx()).GetBondTypeAsDouble()==2.0
                and n.GetAtomicNum()==6 and not n.GetIsAromatic() for n in nbrs)
            if has_dbl_C and not has_H:
                carbonyl_o.append(idx)
            if has_arom_C and has_H:
                phenol_o.append(idx)

    # Place Mn at first bidentate site
    bidentate = sorted([(ci, pi, np.linalg.norm(pos_d[ci]-pos_d[pi]))
                         for ci in carbonyl_o for pi in phenol_o
                         if np.linalg.norm(pos_d[ci]-pos_d[pi]) < 3.5],
                        key=lambda x: x[2])

    if bidentate:
        ci, pi, d_oo = bidentate[0]
        o1, o2 = pos_d[ci], pos_d[pi]
        mid = (o1+o2)/2
        ring_c = np.array([pos_d[i] for i,s in enumerate(sym_d) if s=='C']).mean(0)
        oo_unit = (o2-o1)/np.linalg.norm(o2-o1)
        raw = mid - ring_c
        out = raw - np.dot(raw, oo_unit)*oo_unit
        out /= np.linalg.norm(out)
        d_perp = np.sqrt(max(0., 2.1**2 - (d_oo/2)**2))
        mn_pos = mid + d_perp * out
    else:
        mn_pos = np.array([0., 0., 3.0])

    # Remove ionizable H
    ionizable_h = set()
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 1:
            nbrs = list(atom.GetNeighbors())
            if len(nbrs)==1 and nbrs[0].GetAtomicNum()==8:
                o = nbrs[0]; o_nbrs = list(o.GetNeighbors())
                ph = any(n.GetAtomicNum()==6 and n.GetIsAromatic() for n in o_nbrs)
                co = any(n.GetAtomicNum()==6 and not n.GetIsAromatic() and
                         any(mol.GetBondBetweenAtoms(n.GetIdx(),nn.GetIdx()).GetBondTypeAsDouble()==2.0
                             and nn.GetAtomicNum()==8 for nn in n.GetNeighbors() if nn!=o)
                         for n in o_nbrs)
                if ph or co:
                    ionizable_h.add(atom.GetIdx())

    keep = [i for i in range(len(sym_d)) if i not in ionizable_h]
    d4_sym = [sym_d[i] for i in keep]; d4_pos = pos_d[keep]
    dobdc4 = Atoms(d4_sym, positions=d4_pos)
    mn_at  = Atoms(["Mn"], positions=[mn_pos])
    atoms_cluster = dobdc4 + mn_at

    # Pack into periodic box — 20 Å to avoid image interactions
    BOX = 20.0
    atoms_mof = atoms_cluster.copy()
    atoms_mof.set_cell([BOX, BOX, BOX])
    atoms_mof.center()
    atoms_mof.set_pbc([True, True, True])

    print(f"Simplified model: {len(atoms_mof)} atoms in {BOX:.0f}³ Å box")
    print(f"  (This is a Mn-DOBDC⁴⁻ fragment in vacuum with PBC — not real MOF-74)")

atoms_mof.write(f"{WORK_DIR}/mof74_mn_structure.xyz")
print(f"\nSaved → {WORK_DIR}/mof74_mn_structure.xyz")


In [ ]:
sym  = atoms_mof.get_chemical_symbols()
pos  = atoms_mof.get_positions()
cell = atoms_mof.get_cell()

from collections import Counter
comp = Counter(sym)
print("── Structure Summary ───────────────────────────────────────")
print(f"  Total atoms  : {len(atoms_mof)}")
print(f"  Composition  : {dict(comp)}")
print(f"  PBC          : {atoms_mof.get_pbc().tolist()}")
print()
lengths = cell.lengths(); angles = cell.angles()
print(f"  a = {lengths[0]:.3f} Å    α = {angles[0]:.2f}°")
print(f"  b = {lengths[1]:.3f} Å    β = {angles[1]:.2f}°")
print(f"  c = {lengths[2]:.3f} Å    γ = {angles[2]:.2f}°")
print(f"  Volume  : {cell.volume:.2f} Å³")
print()
n_mn = comp.get("Mn", 0)
print(f"  Mn atoms per cell : {n_mn}")
if n_mn > 0:
    # Mn–Mn distances
    mn_pos_list = [pos[i] for i,s in enumerate(sym) if s=="Mn"]
    if len(mn_pos_list) > 1:
        mn_arr = np.array(mn_pos_list)
        for i in range(len(mn_arr)):
            for j in range(i+1, len(mn_arr)):
                d = np.linalg.norm(mn_arr[i]-mn_arr[j])
                if d < 5.0:
                    print(f"  Mn–Mn: {d:.3f} Å")


### 2.1 · 3D Visualisation with Unit Cell Box

In [ ]:
if not HAS_PY3DMOL:
    print("py3Dmol not available"); raise SystemExit

_syms = atoms_mof.get_chemical_symbols()
_pos  = atoms_mof.get_positions()
_cell = atoms_mof.get_cell()

# Bond detection (PBC-aware NeighborList)
_rc = [VDW.get(s, 0.8) for s in _syms]
_nl = NeighborList(_rc, skin=0.25, self_interaction=False, bothways=False)
_nl.update(atoms_mof)

view = py3Dmol.view(width=750, height=520)
view.setBackgroundColor("#0d0d1a")

# Atoms
for i, (sym, p) in enumerate(zip(_syms, _pos)):
    col = CPK.get(sym, "#AAAAAA")
    rad = VDW.get(sym, 0.8) * (0.50 if sym in ("Mn","Fe","Co") else 0.28)
    view.addSphere({"center":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                    "radius":rad, "color":col, "opacity":1.0})

# Bonds (molecular bonds, no PBC wrapping in py3Dmol)
for i in range(len(_syms)):
    idxs, _ = _nl.get_neighbors(i)
    for j in idxs:
        p1, p2 = _pos[i], _pos[j]
        mid3 = (p1+p2)/2
        c1, c2 = CPK.get(_syms[i],"#888"), CPK.get(_syms[j],"#888")
        for (st, en, col) in [(p1, mid3, c1), (mid3, p2, c2)]:
            view.addCylinder({"start":{"x":float(st[0]),"y":float(st[1]),"z":float(st[2])},
                              "end":  {"x":float(en[0]),"y":float(en[1]),"z":float(en[2])},
                              "radius":0.06, "color":col, "opacity":0.9})

# ── Unit cell box ─────────────────────────────────────────────────────────────
# Corners: index 0..7 where bit 2 = along a, bit 1 = along b, bit 0 = along c
_a, _b, _c = np.array(_cell[0]), np.array(_cell[1]), np.array(_cell[2])
_orig = np.zeros(3)

def _corner(ia, ib, ic):
    return _orig + ia*_a + ib*_b + ic*_c

_corners = [_corner((i>>2)&1, (i>>1)&1, i&1) for i in range(8)]
_edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

for ei, ej in _edges:
    p1, p2 = _corners[ei], _corners[ej]
    view.addCylinder({
        "start":{"x":float(p1[0]),"y":float(p1[1]),"z":float(p1[2])},
        "end":  {"x":float(p2[0]),"y":float(p2[1]),"z":float(p2[2])},
        "radius":0.04, "color":"#88AAEE", "opacity":0.7
    })

# Mn labels
for i, (sym, p) in enumerate(zip(_syms, _pos)):
    if sym == "Mn":
        view.addLabel("Mn", {"position":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                              "fontColor":"white","backgroundColor":"transparent",
                              "fontSize":11})

view.zoomTo(); view.show()


## Section 3 · AtomicGraph (Periodic)

For periodic systems, `AtomicGraph.from_ase` automatically:
- Sets `pbc = True` along all periodic directions
- Adds **unit_shifts** to each edge (the integer vector of PBC image crossings)
- The GOAL models use `unit_shifts` to reconstruct the correct Cartesian displacement
  with the current cell vectors — essential for stress tensor calculations.


In [ ]:
graph = AtomicGraph.from_ase(atoms_mof, cutoff=CUTOFF)

print(f"AtomicGraph  (periodic={'✓' if graph.pbc.any() else '✗'})")
print(f"  Nodes (atoms)    : {graph.num_nodes}")
print(f"  Edges (pairs)    : {graph.num_edges}")
print(f"  edge_index shape : {graph.edge_index.shape}")
print(f"  unit_shifts shape: {graph.unit_shifts.shape}")
print(f"  cell shape       : {graph.cell.shape}")
print(f"  PBC              : {graph.pbc.tolist()}")

# How many edges cross a cell boundary?
n_cross = int((graph.unit_shifts.abs().sum(-1) > 0).sum())
print(f"\n  Cross-cell edges : {n_cross} / {graph.num_edges} "
      f"({100*n_cross/graph.num_edges:.1f}%)")
print("  → Non-zero unit_shifts confirm PBC is active")


In [ ]:
# Recompute edge vectors for analysis (graph stores unit_shifts, not vectors)
cell_mat = graph.cell.squeeze(0).numpy()            # (3,3) row vectors
pos_np   = graph.positions.numpy()                  # (N,3)
ei       = graph.edge_index.numpy()                 # (2,E)
us       = graph.unit_shifts.numpy()                # (E,3) integer

edge_vecs  = pos_np[ei[1]] - pos_np[ei[0]] + us @ cell_mat
edge_dists = np.linalg.norm(edge_vecs, axis=-1)

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_facecolor("#1a1a2e"); fig.patch.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC")
ax.set_title("Edge-length distribution (AtomicGraph, PBC active)", color="#CCCCCC")
ax.hist(edge_dists, bins=60, color="#9C3E8F", alpha=0.8, edgecolor="#333355")
ax.axvline(CUTOFF, color="red", lw=1.5, linestyle="--", label=f"cutoff {CUTOFF} Å")
ax.set_xlabel("Edge length (Å)", color="#CCCCCC")
ax.set_ylabel("Count", color="#CCCCCC")
ax.legend(facecolor="#2a2a4e", labelcolor="white")
plt.tight_layout(); plt.show()
print(f"Edge distances: min {edge_dists.min():.3f} Å  mean {edge_dists.mean():.3f} Å  max {edge_dists.max():.3f} Å")


## Section 4 · MACE-MP-0 Periodic Calculation

MACE-MP-0 was trained on the Materials Project database — a dataset of periodic
crystalline materials. It naturally supports PBC and can compute:
- **Energy**: total potential energy of the unit cell (eV)
- **Forces**: −∂E/∂r for each atom (eV/Å)
- **Stress tensor**: −(1/V) ∂E/∂ε (eV/Å³ = GPa / 160.2)

The stress tensor is important for:
- Geometry optimisation at constant pressure (NPT)
- Predicting mechanical properties (bulk modulus, elastic constants)


In [ ]:
if not HAS_MACE:
    raise RuntimeError("Install MACE: pip install mace-torch")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
mace_calc = mace_mp(model=MACE_MODEL, dispersion=False, device=device,
                    default_dtype="float32")
print(f"MACE-MP-0 ({MACE_MODEL}) loaded on {device}")


In [ ]:
atoms_mace = atoms_mof.copy()
atoms_mace.calc = mace_calc

e_mace  = atoms_mace.get_potential_energy()
f_mace  = atoms_mace.get_forces()
st_mace = atoms_mace.get_stress(voigt=False)   # (3,3) Cartesian stress

vol   = atoms_mace.get_volume()
GPa   = 160.2177                              # 1 eV/Å³ = 160.22 GPa
st_GPa = st_mace * GPa

print(f"MACE-MP-0  single-point (periodic)")
print(f"  Energy         : {e_mace:.4f} eV")
print(f"  Energy/atom    : {e_mace/len(atoms_mace):.4f} eV/atom")
print(f"  Volume         : {vol:.2f} Å³")
print(f"  |F|max         : {np.abs(f_mace).max():.4f} eV/Å")
print(f"  |F|rms         : {np.sqrt((f_mace**2).mean()):.4f} eV/Å")
print()
print("  Stress tensor (GPa):")
for row in st_GPa:
    print(f"    {row[0]:8.3f} {row[1]:8.3f} {row[2]:8.3f}")
print(f"  Hydrostatic pressure: {-np.trace(st_GPa)/3:.3f} GPa")


In [ ]:
# ── Geometry optimisation (NVT: fixed cell, relax atom positions) ─────────────
print("Running BFGS geometry optimisation with MACE (fixed cell) …")
atoms_opt = atoms_mof.copy()
atoms_opt.calc = mace_calc

# Use BFGS for NVT (constant volume) optimisation
opt = BFGS(atoms_opt, logfile=f"{WORK_DIR}/mof74_bfgs.log")
opt.run(fmax=FMAX, steps=400)

e_opt = atoms_opt.get_potential_energy()
f_opt = atoms_opt.get_forces()

print(f"\nAfter BFGS:")
print(f"  Energy        : {e_opt:.4f} eV  (was {e_mace:.4f} eV)")
print(f"  ΔE            : {e_opt - e_mace:.4f} eV")
print(f"  |F|max        : {np.abs(f_opt).max():.4f} eV/Å  (target {FMAX})")

atoms_opt.write(f"{WORK_DIR}/mof74_mn_mace_opt.xyz")
print(f"\nSaved → {WORK_DIR}/mof74_mn_mace_opt.xyz")
atoms_relax = atoms_opt.copy()   # use relaxed structure for subsequent analysis


## Section 5 · GFN2-xTB Periodic Electronic Structure

`xtb-python` supports **periodic boundary conditions** natively via the
`xtb.interface.Calculator` with `lattice` and `periodic` arguments.

This enables:
- Semi-empirical total energy for the periodic cell
- Orbital eigenvalues → approximate band-gap estimate
- Mulliken/Wiberg charges for each site

> **Note on Mn²⁺ spin state in the bulk:**
> The full MOF-74-Mn unit cell contains multiple Mn²⁺ centres. Each has 5
> unpaired electrons, but they can couple ferromagnetically or antiferromagnetically.
> For a quick estimate, we use `uhf = 5 × n_Mn` (ferromagnetic limit). This is an
> approximation; rigorous magnetic ordering requires broken-symmetry DFT.


In [ ]:
if not HAS_XTB_IFACE:
    print("xtb.interface not available — skipping periodic xTB"); raise SystemExit

import ase.units as units

pos_bohr = atoms_relax.get_positions() / units.Bohr
numbers  = atoms_relax.get_atomic_numbers()
cell_mat = atoms_relax.get_cell() / units.Bohr    # cell in Bohr
pbc      = atoms_relax.get_pbc()

n_mn   = int((numbers == 25).sum())               # 25 = Mn
uhf_bulk = 5 * n_mn                               # ferromagnetic estimate

# Formal charge per cell: Mn₂(DOBDC) → each Mn is +2, DOBDC is -4 → net 0
# In practice, use charge = 0 for the neutral unit cell
charge_cell = 0

print(f"Periodic GFN2-xTB:")
print(f"  Atoms in cell : {len(numbers)}")
print(f"  Mn atoms      : {n_mn}")
print(f"  UHF (FM limit): {uhf_bulk}")
print(f"  Charge        : {charge_cell}")

calc_p = XTBInterface(Param.GFN2xTB, numbers, pos_bohr,
                      charge=float(charge_cell), uhf=uhf_bulk,
                      lattice=cell_mat, periodic=pbc)
calc_p.singlepoint()

e_xtb_bulk = calc_p.get_energy() * units.Hartree   # eV
print(f"\nxTB periodic energy: {e_xtb_bulk:.4f} eV")
print(f"  per atom         : {e_xtb_bulk/len(numbers):.4f} eV/atom")

# Orbital eigenvalues (attempt band-gap estimate)
orb_eigs_bulk = calc_p.get_orbital_eigenvalues() * units.Hartree   # eV
if orb_eigs_bulk.ndim == 2:
    alpha_occ = int((numbers.sum() + uhf_bulk) / 2)
    beta_occ  = int((numbers.sum() - uhf_bulk) / 2)
    ae = np.sort(orb_eigs_bulk[0])
    be = np.sort(orb_eigs_bulk[1])
    homo_bulk = ae[alpha_occ - 1]
    lumo_bulk = be[beta_occ]
else:
    eigs_s = np.sort(orb_eigs_bulk)
    n_occ = int(numbers.sum() / 2)
    homo_bulk = eigs_s[n_occ - 1]
    lumo_bulk = eigs_s[n_occ]

gap_bulk = lumo_bulk - homo_bulk
print(f"  HOMO (approx)  : {homo_bulk:+.4f} eV")
print(f"  LUMO (approx)  : {lumo_bulk:+.4f} eV")
print(f"  Gap  (approx)  : {gap_bulk:.4f} eV")


## Section 6 · Electronic Structure and Band Gap

### Important caveat

The **"gap"** computed from xTB orbital eigenvalues in a periodic system is
**not** the same as the crystal band gap:

| Quantity | Method | Meaning |
|---|---|---|
| HOMO-LUMO gap (this notebook) | xTB orbital eigs at Γ-point | Approximate; underestimates true gap |
| Band gap (valence band max → conduction band min) | Periodic DFT with k-point sampling | Physically correct; needs VASP/CP2K/QE |

For MOF-74-Mn, the experimental optical band gap is **~2.3 eV** (from diffuse
reflectance spectroscopy). The xTB gap here will be qualitatively indicative only.

For accurate band gaps, you would use:
```bash
# CP2K (open-source periodic DFT):
#   keyword: PRINT / BANDSTRUCTURE
# VASP:
#   ICHARG=11 non-self-consistent run after SCF
# Quantum ESPRESSO:
#   bands.x post-processing
```


In [ ]:
# ── Density of States (DOS) estimate from GFN2-xTB orbital eigenvalues ───────
eigs_plot = np.sort(orb_eigs_bulk[0] if orb_eigs_bulk.ndim == 2 else orb_eigs_bulk)

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_facecolor("#1a1a2e"); fig.patch.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC")
ax.set_title("GFN2-xTB Orbital Energies (Γ-point only — not a true DOS)",
             color="#CCCCCC")

# Gaussian-broadened DOS
e_range = np.linspace(eigs_plot.min()-1, eigs_plot.max()+1, 500)
sigma   = 0.10   # eV broadening
dos = sum(np.exp(-0.5*((e_range - e)**2)/sigma**2) for e in eigs_plot)

ax.fill_between(e_range, dos, alpha=0.4, color="#9C3E8F", label="All orbitals")
ax.axvline(homo_bulk, color="#FFD700", lw=2, label=f"HOMO {homo_bulk:.2f} eV")
ax.axvline(lumo_bulk, color="#FF6B6B", lw=2, label=f"LUMO {lumo_bulk:.2f} eV")
ax.axvspan(homo_bulk, lumo_bulk, alpha=0.15, color="white", label=f"Gap ≈ {gap_bulk:.2f} eV")

ax.set_xlabel("Energy (eV)", color="#CCCCCC")
ax.set_ylabel("DOS (arb. units)", color="#CCCCCC")
ax.legend(facecolor="#2a2a4e", labelcolor="white")
plt.tight_layout(); plt.show()


## Section 7 · Pore and Structural Analysis

In [ ]:
# ── Pore geometry analysis ────────────────────────────────────────────────────
#
# For MOF-74-Mn, the characteristic pore diameter is ~11 Å.
# We can estimate it from the crystal structure by finding the largest sphere
# that fits in the void, or from the Mn–Mn inter-rod distance.

sym_arr = atoms_relax.get_chemical_symbols()
pos_arr = atoms_relax.get_positions()

# Mn–Mn distances (across the pore)
mn_idx  = [i for i, s in enumerate(sym_arr) if s == "Mn"]
mn_pos  = pos_arr[mn_idx]

print("── Structural Analysis ────────────────────────────────────")
print(f"  Total atoms  : {len(atoms_relax)}")

if len(mn_idx) >= 2:
    from itertools import combinations
    mn_dists = sorted(
        [(i, j, np.linalg.norm(mn_pos[i]-mn_pos[j]))
         for i, j in combinations(range(len(mn_idx)), 2)],
        key=lambda x: x[2]
    )
    print(f"  Mn count     : {len(mn_idx)}")
    print(f"  Shortest Mn–Mn: {mn_dists[0][2]:.3f} Å")
    print(f"  Mn–Mn distances (up to 8 Å):")
    for (i, j, d) in mn_dists:
        if d < 8.0:
            print(f"    Mn{mn_idx[i]}–Mn{mn_idx[j]}: {d:.3f} Å")
elif len(mn_idx) == 1:
    print(f"  Single Mn in cell (simplified model)")
    # Estimate pore from max distance between atoms
    max_span = np.ptp(pos_arr, axis=0)
    print(f"  Molecular span: {max_span}")

# Coordination number analysis for Mn
print()
for mi in mn_idx[:3]:   # first 3 Mn atoms
    mn_p = pos_arr[mi]
    coord_atoms = []
    for j, (s, p) in enumerate(zip(sym_arr, pos_arr)):
        if j != mi:
            d = np.linalg.norm(p - mn_p)
            if d < 2.6:   # Mn–O typical < 2.5 Å
                coord_atoms.append((s, d))
    coord_atoms.sort(key=lambda x: x[1])
    print(f"  Mn({mi}) coordination ({len(coord_atoms)} bonds):")
    for (s, d) in coord_atoms:
        print(f"    Mn–{s}: {d:.3f} Å")

# Volume per Mn (crude estimate of pore volume)
vol  = atoms_relax.get_volume()
n_mn = len(mn_idx) if mn_idx else 1
print(f"\n  Cell volume      : {vol:.2f} Å³")
print(f"  Volume/Mn        : {vol/n_mn:.2f} Å³/Mn  (crude proxy for pore volume)")


## Summary

In [ ]:
print("═"*60)
print("  SUMMARY: MOF-74-Mn Bulk Periodic Structure")
print("═"*60)
print(f"  Atoms in cell    : {len(atoms_mof)}")
print(f"  PBC              : {atoms_mof.get_pbc().tolist()}")
lengths = atoms_mof.cell.lengths(); angles = atoms_mof.cell.angles()
print(f"  a,b,c (Å)       : {lengths[0]:.2f}  {lengths[1]:.2f}  {lengths[2]:.2f}")
print(f"  Volume           : {atoms_mof.get_volume():.2f} Å³")
print()
print(f"  MACE energy      : {e_mace:.4f} eV")
print(f"  MACE/atom        : {e_mace/len(atoms_mof):.4f} eV/atom")
print(f"  MACE stress (hyd): {-np.trace(st_mace * 160.2)/3:.3f} GPa")
print(f"  xTB energy       : {e_xtb_bulk:.4f} eV")
print(f"  Gap (Γ, xTB)     : {gap_bulk:.4f} eV")
print(f"    (Note: use periodic DFT for accurate band gap)")
print()
import glob
for f in sorted(glob.glob(f"{WORK_DIR}/*")):
    print(f"  {f}")
print("═"*60)
